[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_08/ATS_ch8_jumps/ATS_ch8_jumps.ipynb)

# ATS_ch8_jumps

Bipower variation, tripower quarticity and the ratio jump test of Huang and Tauchen (2005): size and power by simulation with 5-minute and 1-minute returns; jump days of Bitcoin and Ether from 5-minute returns, 2018-2026, against the number of false rejections expected under the null.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 8: Advanced volatility modelling: realised measures, HAR and multivariate GARCH. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_8'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
# local copy of the Oxford-Man realized library, if present (not distributed; read_omi downloads it)
REALIZED_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                     if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')
END = '2026-09-18'          # last day of the saved data
# Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and Sheppard 2009): the last public
# file (28 February 2022), preserved by the Internet Archive; six indices and the main measures
OMI_URL = ('https://web.archive.org/web/20220301022212id_/'
           'https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip')
OMI_FILE = 'omi_realized_library_v03.csv'
OMI_SYMBOLS = ['.SPX', '.GDAXI', '.FTSE', '.N225', '.STOXX50E', '.FCHI']
OMI_COLS = ['rv5', 'rv5_ss', 'rv10', 'bv', 'bv_ss', 'medrv', 'rk_parzen', 'rk_twoscale', 'rsv', 'open_to_close',
            'close_price', 'open_price', 'nobs']

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_omi(symbols=None, local=True):
    """Oxford-Man Institute's realized library, version 0.3: daily realised measures (OMI_COLS) of six indices
    (OMI_SYMBOLS), columns date, symbol and the measures, sorted by symbol and date. The library is not
    redistributed with the course: a local copy data/realized/omi_realized_library_v03.csv is used if present,
    otherwise the last public file (Internet Archive copy of 28 February 2022) is downloaded once, the six indices
    are extracted and cached in the temporary folder. Cite: Heber, G., Lunde, A., Shephard, N. and Sheppard, K.
    (2009), Oxford-Man Institute's realized library, version 0.3, Oxford-Man Institute, University of Oxford."""
    import tempfile
    import zipfile
    if 'omi' not in _CACHE:
        cache = os.path.join(tempfile.gettempdir(), 'ats_omi')
        cands = ([os.path.join(REALIZED_DIR, OMI_FILE)] if (local and REALIZED_DIR) else []) + [os.path.join(cache, OMI_FILE)]
        path = next((c for c in cands if os.path.exists(c)), '')
        if not path:
            print("Downloading the Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and "
                  "Sheppard 2009), last public file of 28 February 2022, from the Internet Archive")
            os.makedirs(cache, exist_ok=True)
            zpath = os.path.join(cache, 'oxfordmanrealizedvolatilityindices.zip')
            if not os.path.exists(zpath):
                req = urllib.request.Request(OMI_URL, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=300) as r:
                    raw = r.read()
                with open(zpath, 'wb') as f:
                    f.write(raw)
            with zipfile.ZipFile(zpath) as z:
                d = pd.read_csv(z.open(z.namelist()[0]))
            d = d.rename(columns={d.columns[0]: 'date'})
            d['date'] = pd.to_datetime(d['date'].str[:10])
            d = d[d['Symbol'].isin(OMI_SYMBOLS)][['date', 'Symbol'] + OMI_COLS].rename(columns={'Symbol': 'symbol'})
            path = os.path.join(cache, OMI_FILE)
            d.sort_values(['symbol', 'date']).to_csv(path, index=False, float_format='%.6g')
        _CACHE['omi'] = pd.read_csv(path, parse_dates=['date'])
    d = _CACHE['omi']
    if symbols is not None:
        d = d[d['symbol'].isin([symbols] if isinstance(symbols, str) else list(symbols))]
    return d.copy()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import time

from scipy import special

from scipy.signal import lfilter

SEED = 2026

REAL_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/realized/'

REAL_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                 if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')

OMI_NAMES = {'.SPX': 'S&P 500', '.GDAXI': 'DAX', '.FTSE': 'FTSE 100', '.N225': 'Nikkei 225', '.STOXX50E': 'Euro Stoxx 50', '.FCHI': 'CAC 40'}

MU1 = np.sqrt(2 / np.pi)

HT_C = np.pi ** 2 / 4 + np.pi - 5

SIM = {'n': 23400, 'mean_iv': 1.0, 'sd_slow': 0.75, 'sd_fast': 0.45, 'hl_slow': 60.0, 'hl_fast': 2.0, 'rho': -0.6, 'lam': 0.08, 'sd_jump': 0.8, 'omega': 0.004}

HAR_WIN = 1000

GMV_ASSETS = ['AAPL.US', 'AMZN.US', 'BAC.US', 'C.US', 'GOOGL.US', 'GS.US', 'JPM.US', 'META.US', 'MS.US', 'MSFT.US', 'MSTR.US', 'NVDA.US', 'TSLA.US', 'WFC.US', 'SPY.US', 'QQQ.US', 'RSP.US']

def save(name, save_it=True):
    st.check_no_grey(plt.gcf())
    if save_it:
        st.save_fig(name)
    else:
        plt.show()
        plt.close()


def read_realized(fname, **kw):
    """A file of data/realized: local copy of the course data, otherwise the ATS repository on GitHub."""
    p = os.path.join(REAL_DIR, fname) if REAL_DIR else ''
    return pd.read_csv(p if p and os.path.exists(p) else REAL_RAW + fname, **kw)


def omi(symbol='.SPX'):
    """Oxford-Man realized library v0.3 (Heber, Lunde, Shephard and Sheppard 2009): one index, daily, returns in %
    and variances in %^2; days with a non-positive measure are dropped."""
    d = read_omi()
    d = d[d['symbol'] == symbol].set_index('date').sort_index()
    out = pd.DataFrame({'rv': 1e4 * d['rv5'], 'rv_ss': 1e4 * d['rv5_ss'], 'bv': 1e4 * d['bv'],
                        'medrv': 1e4 * d['medrv'], 'rk': 1e4 * d['rk_parzen'], 'rsv': 1e4 * d['rsv'],
                        'oc': 100 * d['open_to_close'], 'nobs': d['nobs']})
    return out[(out[['rv', 'bv', 'rk', 'medrv']] > 0).all(axis=1)].dropna()


def binance_daily(coin='btc'):
    """Our daily realised measures of Bitcoin or Ether from Binance one-minute prices (UTC days, %, %^2)."""
    d = read_realized('binance_daily_realized.csv', parse_dates=['date'], index_col='date')
    c = [x for x in d.columns if x.startswith(coin + '_')]
    out = d[c].rename(columns=lambda x: x[len(coin) + 1:]).dropna()
    out = out[(out[['rv5', 'bv5', 'rk1']] > 0).all(axis=1)]
    if 'rcov5' in d.columns:
        out['rcov5'] = d['rcov5'].reindex(out.index)
    return out


def returns(name, start='2000-01-01', end=None):
    """Daily log returns in %."""
    p = load_close(name, start=start, end=end) if end else load_close(name, start=start)
    return (100 * np.log(p).diff()).dropna()


def simulate_intraday(days, rng=None, p=SIM, jumps=True, noise=True):
    """Calibrated two-factor log-SV with leverage, compound Poisson jumps and i.i.d. Gaussian noise, one-second grid
    (n seconds a day, returns in %). Returns efficient log prices X (days x n+1), observed Y, IV and QV per day."""
    rng = rng or np.random.default_rng(SEED)
    n = p['n']
    N = days * n
    dt = 1.0 / n
    phis = [np.exp(-np.log(2) / (p[h] * n)) for h in ('hl_slow', 'hl_fast')]
    sds = [p['sd_slow'], p['sd_fast']]
    eps_p = rng.standard_normal(N)
    w_f = p['rho'] * eps_p + np.sqrt(1 - p['rho'] ** 2) * rng.standard_normal(N)
    w_s = rng.standard_normal(N)
    fac = []
    for ph, sd, w in zip(phis, sds, (w_s, w_f)):
        x0 = sd * rng.standard_normal()
        f = lfilter([sd * np.sqrt(1 - ph ** 2)], [1.0, -ph], w, zi=[ph * x0])[0]
        fac.append(f)
    logv = np.log(p['mean_iv']) - 0.5 * (sds[0] ** 2 + sds[1] ** 2) + fac[0] + fac[1]
    sig2 = np.exp(logv)                       # spot variance in %^2 per day
    dX = np.sqrt(sig2 * dt) * np.r_[eps_p[1:], eps_p[:1]]   # price shock of t+1 correlated with vol shock of t
    iv = (sig2 * dt).reshape(days, n).sum(1)
    J = np.zeros(N)
    if jumps:
        nj = rng.poisson(p['lam'] * days)
        pos = rng.integers(0, N, nj)
        np.add.at(J, pos, p['sd_jump'] * rng.standard_normal(nj))
    dX = (dX + J).reshape(days, n)
    X = np.hstack([np.zeros((days, 1)), np.cumsum(dX, 1)])
    Y = X + (p['omega'] * rng.standard_normal(X.shape) if noise else 0.0)
    qv = iv + (J.reshape(days, n) ** 2).sum(1)
    return X, Y, iv, qv


def rv_k(P, k):
    """Realised variance of the log-price rows P sampled every k seconds."""
    r = np.diff(P[:, ::k], axis=1)
    return (r ** 2).sum(1), r


def rv_subsampled(P, k):
    """Average of the k offset grids (subsampling), each rescaled to the full day."""
    n = P.shape[1] - 1
    out = np.zeros(P.shape[0])
    for o in range(k):
        r = np.diff(P[:, o::k], axis=1)
        out += (r ** 2).sum(1) * n / (n - o) / k
    return out


def tsrv(P, K):
    """Two-scales realised variance (Zhang, Mykland and Ait-Sahalia 2005): RV_avg(K) - (n_bar / n) RV_all."""
    n = P.shape[1] - 1
    nbar = (n - K + 1) / K
    rv_all = (np.diff(P, axis=1) ** 2).sum(1)
    rv_avg = np.mean([(np.diff(P[:, o::K], axis=1) ** 2).sum(1) for o in range(K)], axis=0)
    return (rv_avg - nbar / n * rv_all) / (1 - nbar / n)


def parzen(x):
    x = np.abs(x)
    return np.where(x <= 0.5, 1 - 6 * x ** 2 + 6 * x ** 3, np.where(x <= 1, 2 * (1 - x) ** 3, 0.0))


def realised_kernel(r, H):
    """Realised kernel with Parzen weights (Barndorff-Nielsen, Hansen, Lunde and Shephard 2008), one day of returns."""
    rk = r @ r
    for h in range(1, H + 1):
        rk += 2 * parzen(h / (H + 1)) * (r[h:] @ r[:-h])
    return rk


def rk_bandwidth(r, r_sparse):
    """BNHLS (2009) rule H = 3.5134 xi^(4/5) n^(3/5), xi^2 = omega^2 / IV, omega^2 = RV/(2n), IV from sparse RV."""
    n = len(r)
    omega2 = (r @ r) / (2 * n)
    xi2 = omega2 / max(r_sparse @ r_sparse, 1e-12)
    return max(1, int(np.ceil(3.5134 * xi2 ** 0.4 * n ** 0.6)))


def bipower(r):
    """Bipower variation of the rows of r (Barndorff-Nielsen and Shephard 2004), small-sample factor n/(n-1)."""
    n = r.shape[-1]
    return MU1 ** -2 * n / (n - 1) * np.sum(np.abs(r[..., 1:]) * np.abs(r[..., :-1]), -1)


def tripower(r):
    """Realised tripower quarticity, an estimator of the integrated quarticity robust to jumps."""
    from math import gamma
    mu43 = 2 ** (2 / 3) * gamma(7 / 6) / gamma(1 / 2)
    n = r.shape[-1]
    a = np.abs(r) ** (4 / 3)
    return n * mu43 ** -3 * n / (n - 2) * np.sum(a[..., 2:] * a[..., 1:-1] * a[..., :-2], -1)


def ht_stat(rv, bv, tq, n):
    """Ratio jump statistic of Huang and Tauchen (2005), max-adjusted; N(0, 1) under no jumps."""
    return ((rv - bv) / rv) / np.sqrt(HT_C / n * np.maximum(1.0, tq / bv ** 2))


def jump_days(d, n, alpha=0.001):
    z = ht_stat(d['rv5'], d['bv5'], d['tq5'], n)
    return z, z > stats.norm.ppf(1 - alpha)


def fig_jump_sim(save_it=True, days=2000):
    """Size and power of the Huang-Tauchen test at 0.1% (one-sided), 5-minute and 1-minute returns, with and
    without noise; power as a function of the jump size (in units of the daily volatility)."""
    rng = np.random.default_rng(SEED + 3)
    p = dict(SIM, n=390 * 60)
    X, Y, iv, qv = simulate_intraday(days, rng, p, jumps=False)
    crit = stats.norm.ppf(0.999)
    out = {'crit': float(crit), 'days': days}
    for lab, P in (('clean', X), ('noisy', Y)):
        for k, nm in ((300, '5m'), (60, '1m'), (5, '5s')):
            r = np.diff(P[:, ::k], axis=1)
            z = ht_stat((r ** 2).sum(1), bipower(r), tripower(r), r.shape[1])
            out[f'size_{lab}_{nm}'] = float(np.mean(z > crit))
    sizes = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
    pw = {'5m': [], '1m': []}
    pos = rng.integers(0, X.shape[1] - 1, days)
    sg = np.sign(rng.standard_normal(days))
    for s in sizes:
        Xj = X.copy()
        for d in range(days):
            Xj[d, pos[d] + 1:] += s * sg[d] * np.sqrt(iv[d])
        for k, nm in ((300, '5m'), (60, '1m')):
            r = np.diff(Xj[:, ::k], axis=1)
            z = ht_stat((r ** 2).sum(1), bipower(r), tripower(r), r.shape[1])
            pw[nm].append(float(np.mean(z > crit)))
    out['sizes'] = sizes
    out['power'] = pw
    fig, ax = plt.subplots(figsize=(10, 4.0))
    ax.plot(sizes, pw['5m'], 'o-', color=st.MainBlue, label='5-minute returns (n = 78)')
    ax.plot(sizes, pw['1m'], 's-', color=st.IDAred, label='1-minute returns (n = 390)')
    ax.axhline(0.001, color=st.DarkText, lw=0.8, ls=':')
    ax.set_xlabel('jump size / daily volatility sqrt(IV)')
    ax.set_ylabel('rejection rate at 0.1%')
    st.legend_outside_bottom(ax, ncol=2, y=-0.2)
    save('ats_ch8_jump_power', save_it)
    return out


def fig_jumps_crypto(save_it=True):
    """Jump days of Bitcoin and Ether (5-minute returns, n = 288), share by year; jump share of total variation."""
    out = {}
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
    for coin, c, lab in (('btc', st.COL['btc'], 'Bitcoin'), ('eth', st.Purple, 'Ether')):
        d = binance_daily(coin)
        z, j = jump_days(d, 288)
        jv = np.where(j, np.maximum(d['rv5'] - d['bv5'], 0), 0)
        share = pd.Series(j.values, d.index).groupby(d.index.year).mean()
        axes[0].plot(share.index, 100 * share.values, 'o-', color=c, label=f'{lab}: % of days with a jump at 0.1%')
        axes[1].hist(np.clip(z, -6, 12), bins=60, density=True, histtype='step', color=c, lw=1.5, label=f'{lab}: Huang-Tauchen z')
        out[coin] = dict(T=len(d), share=float(j.mean()), n_jump=int(j.sum()), expected_false=float(0.001 * len(d)),
                         jv_share=float(jv.sum() / d['rv5'].sum()), mean_z=float(z.mean()),
                         first=str(d.index[0].date()), last=str(d.index[-1].date()),
                         rv_bv=float(np.mean(np.maximum(d['rv5'] - d['bv5'], 0)) / d['rv5'].mean()))
    g = np.linspace(-4, 6, 300)
    axes[1].plot(g, stats.norm.pdf(g), color=st.Forest, ls='--', label='N(0, 1)')
    axes[0].set_ylabel('% of days')
    axes[1].set_xlabel('z')
    st.fig_legend_bottom(fig, ncol=3, y=-0.02)
    plt.tight_layout()
    save('ats_ch8_jumps_crypto', save_it)
    return out

## Run

In [ ]:
print(fig_jump_sim(days=500))
print(fig_jumps_crypto())

![ats_ch8_jump_power](./ats_ch8_jump_power.png)

Output: `ats_ch8_jump_power.pdf`

![ats_ch8_jumps_crypto](./ats_ch8_jumps_crypto.png)

Output: `ats_ch8_jumps_crypto.pdf`